# 4.2 · OWL 2

OWL (2004) was tried in the field; OWL 2 (2009) fixed limitations. OWL 2 DL is based on **`SROIQ(D)`** — more expressive than OWL DL's `SHOIN(D)`.

In [1]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))  # repo paths
sys.path.insert(0, '.')
import matplotlib; matplotlib.use('Agg')
import ch4_toolkit as ch4
import owlready2, rdflib, owlrl, pandas as pd
print('owlready2', owlready2.VERSION, '| rdflib', rdflib.__version__, '| toolkit ready')

owlready2 0.50 | rdflib 7.6.0 | toolkit ready


## Limitations of OWL 1 that motivated OWL 2
- **Expressivity:** no *qualified* cardinality (`Bicycle ⊑ ≥2 hasComponent.Wheel`); missing reflexivity/irreflexivity/asymmetry; datatype limits.
- **Syntax:** frame legacy + axioms was confusing; RDF triples hard to read.
- **Semantics:** issues around RDF blank nodes / unnamed individuals.

## 4.2.1 New OWL 2 features (`SROIQ`)

In [2]:
for k, v in ch4.OWL2_NEW_FEATURES.items():
    print(f'• {k}: {v}')

• Qualified cardinality restrictions: ≥ nR.C, ≤ nR.C, = nR.C — e.g. Bicycle ⊑ ≥2 hasComponent.Wheel (the Q in SROIQ).
• Local reflexivity (Self): ∃R.Self — e.g. ∃knows.Self for a Narcissist (ObjectHasSelf).
• Global reflexivity: Ref(R) — everything is connected to itself.
• Irreflexivity: Irr(R) — proper parthood: nothing is a proper part of itself.
• Asymmetry: Asym(R) — Asym(parentOf): if John is parent of Divesh, Divesh is not parent of John.
• Property chains: R ∘ S ⊑ R — e.g. childOf ∘ childOf ⊑ grandchildOf (the R in SROIQ).


### Built in Python
`Bicycle ⊑ ≥2 hasComponent.Wheel` (qualified cardinality), `∃knows.Self` (Narcissist), irreflexive+asymmetric `properPartOf`, and the `hasMother ∘ hasSister ⊑ hasAunt` property chain.

In [3]:
feat = ch4.build_owl2_features()
print('classes  :', [c.name for c in feat.classes()])
print('Bicycle  :', ch4.dl_render(feat.Bicycle))
print('Narcissist:', ch4.dl_render(feat.Narcissist))

classes  : ['Thing_', 'Wheel', 'Component', 'Door', 'Bicycle', 'Narcissist']
Bicycle  : Bicycle ⊑ Thing
Bicycle ⊑ ≥2 hasComponent.Wheel
Narcissist: Narcissist ⊑ Thing
Narcissist ⊑ knows?


### Sidebar 7 — an OWL ontology is **not** the same as an RDF graph
A DL-based OWL ontology has a *direct (model-theoretic) semantics*; it can be *mapped into* an RDF graph for serialisation, but that doesn't give it a graph semantics. (Save ontologies as `.owl` RDF/XML, not `.rdf`/`.ttl`.) We can still view the RDF *serialisation* of our ontology:

In [4]:
fyc = ch4.firstyearcourse_ontology()
g = fyc.world.as_rdflib_graph()
print('triples in the RDF serialisation:', len(g))
print('… but the *ontology* is the SROIQ theory, not these triples.')

triples in the RDF serialisation: 11
… but the *ontology* is the SROIQ theory, not these triples.


## Example 4.2 — cakes & allergies: the *simple object property* trade-off

The ontologist wants **`TransitiveObjectProperty(hasPart)`** (so cake⊃butter⊃milk ⇒ cake has milk) **and** **`ObjectExactCardinality(4 hasPart Ingredient)`** for regular cakes. OWL 2 DL forbids both: a *non-simple* (transitive) property may not carry a cardinality restriction.

In [5]:
cake = ch4.cakes_allergies_demo()
print(cake['transitive_inference'])
print('triples before/after RL closure:', cake['triples_before_after'])
print()
print('CONFLICT:', cake['conflict'])
print()
print('Reasoner error you would see in an ODE:')
print('  ', cake['reasoner_error'])

cake1 hasPart milk1 inferred = True
triples before/after RL closure: (5, 123)

CONFLICT: A DL reasoner rejects asserting BOTH TransitiveObjectProperty(hasPart) AND ObjectExactCardinality(4 hasPart Ingredient): a *non-simple* (transitive) property may not appear in a cardinality restriction. You must choose transitivity OR the qualified cardinality, not both.

Reasoner error you would see in an ODE:
   Non-simple property '<ex#hasPart>' or its inverse appears in the cardinality restriction 'ObjectMaxCardinality(4 <ex#hasPart> <ex#Ingredient>)'.


**Features usable only on *simple* object properties** (no transitive/chained sub-properties):

In [6]:
for f in ch4.SIMPLE_ONLY_FEATURES:
    print('•', f)

• ObjectMinCardinality
• ObjectMaxCardinality
• ObjectExactCardinality
• ObjectHasSelf
• FunctionalObjectProperty
• InverseFunctionalObjectProperty
• IrreflexiveObjectProperty
• AsymmetricObjectProperty
• DisjointObjectProperties


## 4.2.2 OWL 2 Profiles (EL / QL / RL)
Sub-languages of OWL 2 DL for **scalability**. Each has tailored reasoners.

In [7]:
import pandas as pd
pd.DataFrame(ch4.OWL2_PROFILES).T

,dl,complexity,purpose
OWL 2 EL,EL++,PTime-complete,"Large, relatively simple type-level (TBox) ont..."
OWL 2 QL,DL-Lite_R,NLogSpace / AC0 (data),Query answering over large amounts of data (OB...
OWL 2 RL,DLP / pD*,PTime-complete,Rules + data as RDF triples; scalable forward-...


### A lightweight 'OWL Classifier' in Python
The chapter mentions the *OWL Classifier* tool. Here is a small heuristic feature checker that flags constructs pushing an ontology out of the lightweight profiles.

In [8]:
import json
print(json.dumps(ch4.classify_profile(ch4.build_awo(1)), indent=2, default=str))

{
  "feature_counts": {
    "union (\u2294)": 2,
    "complement (\u00ac)": 0,
    "allValuesFrom (\u2200)": 2,
    "cardinality": 0,
    "someValuesFrom (\u2203)": 5,
    "oneOf": 0
  },
  "verdict": [
    "Out of OWL 2 EL (EL++ has no \u2294, \u00ac, \u2200).",
    "\u2203 on LHS / \u2200 anywhere narrows the profile."
  ]
}


## 4.2.3 OWL 2 syntaxes — one axiom, many renderings

`FirstYearCourse ⊑ ∀isTaughtBy.Professor` (Eq. 4.1) in every syntax from the chapter. RDF/XML & Turtle are produced for real; functional/OWL-XML/Manchester are faithful textual twins of Listings 4.3–4.7.

In [9]:
syn = ch4.render_syntaxes()
for name, text in syn.items():
    print('=' * 70)
    print(name)
    print('-' * 70)
    print(text.strip())

DL (Eq. 4.1)
----------------------------------------------------------------------
FirstYearCourse ⊑ ∀isTaughtBy.Professor
RDF/XML (Listing 4.2, required)
----------------------------------------------------------------------
<?xml version="1.0"?>
<rdf:RDF xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#"
         xmlns:xsd="http://www.w3.org/2001/XMLSchema#"
         xmlns:rdfs="http://www.w3.org/2000/01/rdf-schema#"
         xmlns:owl="http://www.w3.org/2002/07/owl#"
         xml:base="http://www.yourpa.ge/ontologies/exUni.owl"
         xmlns="http://www.yourpa.ge/ontologies/exUni.owl#">

<owl:Ontology rdf:about="http://www.yourpa.ge/ontologies/exUni.owl"/>

<owl:ObjectProperty rdf:about="#isTaughtBy"/>

<owl:Class rdf:about="#Professor">
  <rdfs:subClassOf rdf:resource="http://www.w3.org/2002/07/owl#Thing"/>
</owl:Class>

<owl:Class rdf:about="#FirstYearCourse">
  <rdfs:subClassOf rdf:resource="http://www.w3.org/2002/07/owl#Thing"/>
  <rdfs:subClassOf>
    <owl:Restriction>
 

## 4.2.4 Complexity of OWL species (Table 4.3)
Worst-case complexity of the standard reasoning problems.

In [10]:
ch4.complexity_table_4_3()

,Language,Taxonomic,Data,Query,Combined
0,OWL 2 (RDF-based sem.),Undecidable,Undecidable,Undecidable,Undecidable
1,OWL 2 (Direct sem.),2NEXPTIME-complete,Decidable (NP-Hard),N/A,2NEXPTIME-complete
2,OWL 2 EL,PTIME-complete,PTIME-complete,NP-complete (CQ),PSPACE-complete (CQ)
3,OWL 2 QL,NLogSpace-complete,AC0,NP-complete (CQ),NP-complete (CQ)
4,OWL 2 RL,PTIME-complete,PTIME-complete,NP-complete (CQ),NP-complete (CQ)
5,OWL DL (OWL 1),NEXPTIME-complete,Decidable (NP-Hard),N/A,NEXPTIME-complete
